In [86]:
from utils.paths import SAMPLING_DIR, RAW_DIR,PREDICTIONS_DIR

In [87]:
import numpy as np
import pandas as pd
import pickle
from models_scripts.AdaptiveTransferKernel import AdaptiveTransferKernel
from sklearn.preprocessing import StandardScaler

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [108]:
comp_space = pd.read_csv(RAW_DIR/"CuNiAl_descriptors.csv")

Experimental data GP - Random splitting

In [109]:
with open(SAMPLING_DIR/"random_split", "rb") as f:
    random_split = pickle.load(f)

In [119]:
columns = ['Cu','Ni','Al','r','r_ave','del_r','S','del_EN','VEC']

X_train_random = random_split['X_train_split'][columns]
X_test_random = random_split['X_test_split'][columns]

In [120]:
X_train = np.asarray(X_train_random, dtype=float)
y_train = np.asarray(random_split['y_train'], dtype=float).ravel()

x_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X_train)

In [121]:
kernel = (
    C(1.0, (1e-3, 1e3)) *
    Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1))
)

gpr_nt_ran = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,   
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gpr_nt_ran.fit(X_scaled, y_train)
print("Learned kernel:", gpr_nt_ran.kernel_)

Learned kernel: 0.942**2 * Matern(length_scale=2.58, nu=2.5) + WhiteKernel(noise_level=0.26)


In [122]:
gp_random_split = {}
X_test_random = np.asarray(X_test_random, dtype=float)
X_scaled_test = x_scaler.transform(X_test_random)

gp_random_split['y_test_predic'],gp_random_split['std_test_predic'] = gpr_nt_ran.predict(X_scaled_test, return_std=True)


gp_random_split['mae'] = mean_absolute_error(gp_random_split['y_test_predic'], random_split['y_test'])
gp_random_split['rmse'] = np.sqrt(mean_squared_error(gp_random_split['y_test_predic'], random_split['y_test']))
gp_random_split['r2'] = r2_score(gp_random_split['y_test_predic'], random_split['y_test'])


print(f"MAE : {gp_random_split['mae']:.4g}")
print(f"RMSE: {gp_random_split['rmse']:.4g}")
print(f"R^2 : {gp_random_split['r2']:.4g}")

MAE : 5.747
RMSE: 6.729
R^2 : 0.6225


In [124]:
gp_random_split['x_space'] = np.asarray(comp_space[columns], dtype=float)
X_scaled_all = x_scaler.transform(gp_random_split['x_space'])
gp_random_split['y_predic'], gp_random_split['std_y_predic'] = gpr_nt_ran.predict(X_scaled_all, return_std=True)

with open(PREDICTIONS_DIR/"GP-Experiments/gp_random_split", "wb") as f:
    pickle.dump(gp_random_split, f)

Experimental data GP - Kmeans splitting

In [140]:
with open(SAMPLING_DIR/"cluster_split", "rb") as f:
    cluster_split = pickle.load(f)

In [141]:
columns = ['Cu','Ni','Al','r','r_ave','del_r','S','del_EN','VEC']

X_train_cluster = cluster_split['X_train_split'][columns]
X_test_cluster = cluster_split['X_test_split'][columns]


X_train = np.asarray(X_train_cluster, dtype=float)
y_train = np.asarray(cluster_split['y_train'], dtype=float).ravel()

x_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X_train)

In [142]:

kernel = (
    C(1.0, (1e-3, 1e3)) *
    Matern(length_scale=1.0, length_scale_bounds=(1e-2, 1e2), nu=2.5)
    + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-8, 1e1))
)

gpr_nt_clus = GaussianProcessRegressor(
    kernel=kernel,
    alpha=0.0,   
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=42
)

gpr_nt_clus.fit(X_scaled, y_train)
print("Learned kernel:", gpr_nt_clus.kernel_)

Learned kernel: 1.04**2 * Matern(length_scale=2.87, nu=2.5) + WhiteKernel(noise_level=0.216)


In [143]:

gp_cluster_split = {}
X_test_cluster = np.asarray(X_test_cluster, dtype=float)
X_scaled_test = x_scaler.transform(X_test_cluster)

gp_cluster_split['y_test_predic'],gp_cluster_split['std_test_predic'] = gpr_nt_clus.predict(X_scaled_test, return_std=True)


gp_cluster_split['mae'] = mean_absolute_error(gp_cluster_split['y_test_predic'], cluster_split['y_test'])
gp_cluster_split['rmse'] = np.sqrt(mean_squared_error(gp_cluster_split['y_test_predic'], cluster_split['y_test']))
gp_cluster_split['r2'] = r2_score(gp_cluster_split['y_test_predic'], cluster_split['y_test'])


print(f"MAE : {gp_cluster_split['mae']:.4g}")
print(f"RMSE: {gp_cluster_split['rmse']:.4g}")
print(f"R^2 : {gp_cluster_split['r2']:.4g}")


MAE : 9.116
RMSE: 12.24
R^2 : 0.2077


In [148]:

gp_cluster_split['x_space'] = np.asarray(comp_space[columns], dtype=float)
X_scaled_all = x_scaler.transform(gp_cluster_split['x_space'])
gp_cluster_split['y_predic'], gp_cluster_split['std_y_predic'] = gpr_nt_clus.predict(X_scaled_all, return_std=True)

with open(PREDICTIONS_DIR/"GP-Experiments/gp_cluster_split", "wb") as f:
    pickle.dump(gp_cluster_split, f)

Adaptive kernel approach

In [ ]:
# Source and target variables for Adaptive transfer kernel

def data_transfer_gp (X_source,X_target, y_source, y_target):
    # Convert to numpy
    Xs = np.asarray(X_source, dtype=float)
    Xt = np.asarray(X_target, dtype=float)
    
    # Add domain indicator ( 0 = source, 1 = target)
    Xs_aug = np.c_[Xs, np.zeros((len(Xs),1))]
    Xt_aug = np.c_[Xt, np.ones((len(Xt),1))]
    
    #combine datasets
    X_train = np.vstack([Xs_aug, Xt_aug])
    y_train = np.concatenate([y_source, y_target])
    
    return X_train,y_train

In [80]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor

kernel = AdaptiveTransferKernel(
    kernel=1.0 * Matern(length_scale=1.0, nu=2.5),
    lamb=2.0,
    lamb_bounds=(1.0, 3.0),
    different_noises=False,
)

gpr_pe = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-5,            # jitter for numerical stability
    normalize_y=True,      # y standardization inside sklearn
    n_restarts_optimizer=3 # increase later (5-10) if slow/unstable
)